# DROID-SLAM on R&S data (Colab, GPU)

Stereo and mono DROID-SLAM on one sequence, scored with `eval/` next to Jonas's ORB-SLAM3
stereo result (ATE SE3 1.8 mm on seq01). Runtime: **GPU** (T4 is enough; DROID needs ~11 GB
on long clips, 600 frames at 320x512 fit comfortably).

Needs `dataset_extracted/<seq>/` in the mounted Drive. The CUDA build (~15 min) is cached
as wheels in `rs-cache/` on Drive.

In [ ]:
SEQ = "seq01"
BRANCH = "spatial-ai/droid-slam"
DRIVE_DATA = f"/content/drive/MyDrive/dataset_extracted/{SEQ}"
DRIVE_CACHE = "/content/drive/MyDrive/rs-cache"
DRIVE_RESULTS = f"/content/drive/MyDrive/rs-results/{SEQ}"
REPO = "/content/rs"

from google.colab import drive
drive.mount("/content/drive")
import os, torch
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU"
assert os.path.isdir(DRIVE_DATA), f"{DRIVE_DATA} missing: mount the Drive that holds dataset_extracted/"
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))

In [ ]:
%%bash -s "$REPO" "$BRANCH"
set -e
[ -d "$1/.git" ] || git clone -q --branch "$2" https://github.com/AliAlpOezer/rs-hackatum-2026 "$1"
git -C "$1" pull -q; git -C "$1" log --oneline -1

In [ ]:
%%bash -s "$REPO" "$DRIVE_CACHE"
set -e
bash "$1/tracks/spatial-ai/envs/install_droid.sh" /content/DROID-SLAM "$2"
python "$1/tracks/spatial-ai/envs/torch_scatter_shim/test_shim.py"

In [ ]:
%%bash -s "$REPO" "$DRIVE_DATA" "$SEQ"
# Raw R&S -> shared layout (frames/, frames_right/, calib.yaml, gt.txt). Copy locally first: Drive reads are slow.
set -e
cd "$1"
mkdir -p /content/raw && cp -rn "$2" /content/raw/
[ -f data/$3/gt.txt ] || python data/tools/rs_to_seq.py /content/raw/$3 data/$3
ls data/$3 | head; ls data/$3/frames | wc -l

In [ ]:
%%bash -s "$REPO" "$SEQ"
cd "$1"
python tracks/spatial-ai/runners/droid_runner.py data/$2 --droid-dir /content/DROID-SLAM --out results/$2/droid_stereo.txt
python tracks/spatial-ai/runners/droid_runner.py data/$2 --droid-dir /content/DROID-SLAM --mono --out results/$2/droid_mono.txt

In [ ]:
%%bash -s "$REPO" "$SEQ"
# Score next to Jonas's ORB-SLAM3 stereo run (origin/orb-slam-3).
cd "$1"
git fetch -q origin orb-slam-3 && git show origin/orb-slam-3:results/$2/orbslam3_stereo.txt > results/$2/orbslam3_stereo.txt || true
python eval/evaluate.py data/$2/gt.txt results/$2/*.txt --metric | tee results/$2/eval_droid.txt

In [ ]:
%%bash -s "$REPO" "$SEQ" "$DRIVE_RESULTS"
mkdir -p "$3" && cp "$1"/results/$2/droid_* "$1"/results/$2/eval_droid.txt "$3"/ && ls -l "$3"